# B2.1 — RAG: image top-5, all sections (no rerank)

**Pipeline:** `image → EVA-CLIP top-5 articles → pool ALL sections → Qwen2.5-VL` (no rerank, no cap).

Tests whether dumping all raw context beats selecting a few paragraphs. Result: **0.347** — almost as good as B4 (cross-encoder, 0.360), and above B2 (top-1 all sections, 0.295).

In [ ]:
import json, os, re, string
OUT = '../../outputs'
BASE = '/work/cvcs2026/encyclopedic'
preds = [json.loads(l) for l in open(OUT + '/predictions_B2_1.jsonl') if l.strip()]
for label, f in [('A', 'results.json'), ('B2 (top1)', 'results_B2.json'),
                 ('B2.1 (top5)', 'results_B2_1.json'), ('B4 (cross)', 'results_B4.json')]:
    p = OUT + '/' + f
    if os.path.exists(p):
        print('%-14s overall %.4f' % (label, json.load(open(p))['accuracy_overall']))

## Where is the bottleneck? Answer accuracy on retrieval hit vs miss
Split examples by whether the correct article was among the top-5 candidates, then measure answer accuracy on each subset. Correctness uses an exact-match proxy (a floor of the BEM metric), so the absolute numbers are low, but the hit-vs-miss gap is the signal.

In [ ]:
import matplotlib.pyplot as plt
PUNCT = set(string.punctuation + '\u2018\u2019\u00b4`_')

def _pre(a):
    a = a.lower().replace('\n', ' ').replace('\t', ' ').strip()
    a = ''.join('' if c in PUNCT else c for c in a)
    a = re.sub(r'\b(the answer is|a|an|the)\b', ' ', a)
    return ' '.join(a.split())

def correct(rec):
    if rec.get('prediction') is None:
        return False
    ref = str(rec['answer'])
    if rec['question_type'] == 'multi_answer':
        R = [x for x in (_pre(a) for a in ref.split('|')[0].split('&&')) if x]
        C = [x for x in (_pre(a) for a in rec['prediction'].replace(' and ', ',').replace(' & ', ',').split(',')) if x]
        u = len(set(R) | set(C))
        return len(set(R) & set(C)) / u >= 0.5 if u else False
    return _pre(ref) == _pre(rec['prediction'])

def norm(u):
    if not u:
        return u
    u = re.sub(r'^https?://', '', u.strip().lower())
    return u.replace('en.m.wikipedia.org', 'en.wikipedia.org').rstrip('/')

gt = {x['unique_id']: norm(x.get('wikipedia_url'))
      for x in json.load(open(BASE + '/encyclopedic_test_subset.json'))}

def cands(p):
    rc = p.get('retrieved_context') or {}
    cs = [c['wiki_url'] for c in rc.get('candidates', [])] or ([rc['wiki_url']] if rc.get('wiki_url') else [])
    return [norm(u) for u in cs]

hit = [p for p in preds if gt.get(p['unique_id']) in cands(p)]
miss = [p for p in preds if gt.get(p['unique_id']) not in cands(p)]
acc = lambda s: sum(correct(p) for p in s) / len(s) if s else 0
print('recall (article retrieved): %.1f%%' % (100 * len(hit) / len(preds)))
print('accuracy | HIT : %.3f (n=%d)' % (acc(hit), len(hit)))
print('accuracy | MISS: %.3f (n=%d)' % (acc(miss), len(miss)))
plt.figure(figsize=(4, 3.5))
plt.bar(['hit', 'miss'], [acc(hit), acc(miss)])
plt.ylabel('answer accuracy (EM proxy)'); plt.title('B2.1 bottleneck: retrieval')
plt.show()

## Pros / cons

**Pros**
- No reranker model needed, yet nearly matches B4 — for this VLM, dumping all context ≈ cross-encoder selecting a few.
- Beats B2 (top-1 all sections): more candidate articles help.

**Cons / limitations**
- Very long prompts (5 whole articles) → risk of exceeding Qwen's context window; slower generation.
- Most context is from wrong articles (recall ~28%) → noisy.
- Same retrieval ceiling as B3/B4: the hit/miss gap shows retrieval, not generation, is the bottleneck.